# TT-25 — MLP với Keras
## Chấm điểm khách hàng tiềm năng mua bảo hiểm ô tô

Notebook này chạy toàn bộ pipeline mô tả trong `README.md`, gọi trực tiếp các hàm
trong `src/` (không lặp lại logic) để bạn có thể chạy từng bước, xem biểu đồ,
và chỉnh sửa nhanh trước khi đóng gói lại vào `src/train.py`.

**Trước khi chạy:** tải `train.csv` từ Kaggle
([Health Insurance Cross Sell Prediction](https://www.kaggle.com/datasets/anmolkumar/health-insurance-cross-sell-prediction))
và đặt vào `data/train.csv`.

In [ ]:
import sys, os

# Chuyển working directory về thư mục gốc dự án (nơi chứa data/, models/, reports/, src/)
# để mọi đường dẫn trong notebook khớp với khi chạy `python -m src.train` từ gốc dự án.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.data import (
    load_raw, split_data, build_dense_features,
    build_dense_features_with_highcard_onehot, build_embedding_features, TARGET,
)
from src.model import build_mlp, build_mlp_embedding, default_callbacks
from src.evaluate import (
    pr_auc_score, precision_at_k, threshold_for_top_k,
    plot_learning_curves, plot_pr_curve, plot_bar_comparison,
)
from src.train import (
    step_eda, step_baseline_lightgbm, step_mlp_plain_vs_bn_dropout,
    step_architecture_comparison, step_class_weight_comparison,
    step_embedding_comparison, write_conclusion_table, threshold_for_top_k, TOP_K,
)

DATA_PATH = "data/train.csv"
os.makedirs("reports", exist_ok=True)
os.makedirs("models", exist_ok=True)
print("Working directory:", os.getcwd())

## 1. Đọc dữ liệu & EDA nhanh

Kiểm tra tỉ lệ quan tâm (`Response=1`) theo `Previously_Insured`, `Vehicle_Damage` và nhóm tuổi —
đúng như lưu ý trong README: `Previously_Insured` là biến rất mạnh nhưng KHÔNG phải rò rỉ dữ liệu,
vì thông tin này có sẵn trước khi gọi điện.

In [ ]:
df = load_raw(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
eda = step_eda(df)
eda

## 2. Tiền xử lý & chia tập dữ liệu

- `Annual_Premium` được log1p hoá trước khi chuẩn hoá (lệch phải mạnh).
- Các biến phân loại ít mức (`Gender`, `Driving_License`, `Previously_Insured`, `Vehicle_Age`,
  `Vehicle_Damage`) được one-hot.
- Chia train/val/test có `stratify` theo nhãn để giữ đúng tỉ lệ 12,3% quan tâm ở mọi tập.

In [ ]:
df_train, df_val, df_test = split_data(df)
print("train:", df_train.shape, "val:", df_val.shape, "test:", df_test.shape)
print("Tỉ lệ Response trong mỗi tập:",
      df_train[TARGET].mean(), df_val[TARGET].mean(), df_test[TARGET].mean())

(X_train, y_train), (X_val, y_val), (X_test, y_test), _artifacts = build_dense_features(df_train, df_val, df_test)
print("Số đặc trưng dense (one-hot + numeric):", X_train.shape[1])

## 3. Baseline cây — LightGBM

**Bắt buộc theo README**: với dữ liệu dạng bảng, cây gradient boosting thường thắng mạng
nơ-ron. Phải có baseline này để so sánh trung thực trước khi kết luận MLP có đáng dùng không.

In [ ]:
lgbm_result, lgbm_model = step_baseline_lightgbm(X_train, y_train, X_val, y_val, X_test, y_test)
lgbm_result

In [ ]:
display(Image(filename="reports/pr_curve_lightgbm.png"))

## 4. MLP cơ bản vs + Dropout/BatchNorm

So sánh trực tiếp để thấy tác dụng của Dropout + BatchNorm lên PR-AUC và mức độ overfit
(qua learning curves).

In [ ]:
plain_vs_bn_results, best_bn_model = step_mlp_plain_vs_bn_dropout(
    X_train, y_train, X_val, y_val, X_test, y_test
)
pd.DataFrame(plain_vs_bn_results)

In [ ]:
display(Image(filename="reports/learning_curves_plain.png"))
display(Image(filename="reports/learning_curves_bn_dropout.png"))

## 5. So sánh 3 kiến trúc: (64) · (128,64) · (256,128,64)

In [ ]:
arch_results = step_architecture_comparison(X_train, y_train, X_val, y_val, X_test, y_test)
pd.DataFrame(arch_results)

In [ ]:
display(Image(filename="reports/kien_truc_comparison.png"))

## 6. class_weight vs không dùng

Dữ liệu lệch mạnh (~12,3% dương). `class_weight='balanced'` giúp mô hình không "lười" đoán
toàn bộ là lớp đa số — so sánh xem có thực sự cải thiện PR-AUC / Precision@3000 hay không.

In [ ]:
cw_results = step_class_weight_comparison(X_train, y_train, X_val, y_val, X_test, y_test)
pd.DataFrame(cw_results)

## 7. Embedding vs one-hot cho biến nhiều mức

`Region_Code` (53 mức) và `Policy_Sales_Channel` (155 mức) mã hoá bằng số → one-hot tạo ra
200+ cột thưa. Thử thay bằng Embedding layer (kỹ thuật Deep Learning dành riêng cho biến
phân loại nhiều mức) và so sánh PR-AUC cùng số tham số.

In [ ]:
emb_results = step_embedding_comparison(df_train, df_val, df_test)
pd.DataFrame(emb_results)

In [ ]:
display(Image(filename="reports/embedding_vs_onehot.png"))

## 8. Ngưỡng theo Precision@3000 & bảng kết luận cuối cùng

Đội telesales chỉ gọi được 3.000 cuộc/ngày → chọn ngưỡng xác suất tương ứng với việc xếp hạng
top 3.000 khách hàng có khả năng quan tâm cao nhất, thay vì dùng ngưỡng mặc định 0.5.

In [ ]:
import json

proba_best = best_bn_model.predict(X_test, verbose=0).ravel()
threshold = threshold_for_top_k(proba_best, TOP_K)
with open("reports/threshold.json", "w", encoding="utf-8") as f:
    json.dump({"top_k": TOP_K, "probability_threshold": threshold}, f, indent=2)
print(f"Ngưỡng xác suất cho top {TOP_K} khách hàng: {threshold:.4f}")

In [ ]:
all_results = {
    "baseline_vs_best_mlp": [
        lgbm_result,
        max(plain_vs_bn_results + arch_results + cw_results + emb_results, key=lambda r: r["pr_auc"]),
    ],
    "plain_vs_bn": plain_vs_bn_results,
    "architectures": arch_results,
    "class_weight": cw_results,
    "embedding": emb_results,
}
report_path = write_conclusion_table(all_results)
with open(report_path, encoding="utf-8") as f:
    print(f.read())

## Kết luận

Xem bảng đầy đủ tại `reports/comparison_summary.md`. Toàn bộ pipeline này cũng có thể chạy
không cần notebook bằng:

```bash
python -m src.train --data data/train.csv
```